In [2]:
# last
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    get_response_synthesizer,
    StorageContext,
    load_index_from_storage,
)
from llama_parse import LlamaParse
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.ollama import Ollama
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import (
    RelevancyEvaluator,
    CorrectnessEvaluator,
)
import pandas as pd
import asyncio
import json
import time
import nest_asyncio
import os
from types import SimpleNamespace
from llama_index.core.llama_dataset import (
    LabelledRagDataExample,
)
import itertools

# Apply nested asyncio for notebook execution
nest_asyncio.apply()


class RAGEvaluator:
    def __init__(
        self,
        data_directory="data",
        dataset_file="./rag_dataset.json",
        index_persist_dir="./indexes",
    ):
        # Configure settings
        self._configure_settings()

        # Initialize parser and file extractor
        self.parser = LlamaParse(result_type="markdown")
        self.file_extractor = {".pdf": self.parser}

        # Set data directory and persist directory
        self.data_directory = data_directory
        self.index_persist_dir = index_persist_dir

        # Ensure index directory exists
        os.makedirs(self.index_persist_dir, exist_ok=True)

        # Load dataset
        self.rag_dataset = self._load_dataset(dataset_file)

        # Initialize evaluators
        self.relevancy_evaluator = RelevancyEvaluator()
        self.correctness_evaluator = CorrectnessEvaluator()

        # Initialize results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Initialize summary results DataFrame for different parameter combinations
        self.summary_results_df = pd.DataFrame()

    def _configure_settings(self):
        """Configure global settings for embedding model and LLM"""
        Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-zh-v1.5")
        Settings.llm = Ollama(
            model="llama3.1:latest", request_timeout=60.0, temperature=0.3
        )

    def _load_documents(self, data_directory):
        """Load documents from the specified directory"""
        return SimpleDirectoryReader(
            data_directory, file_extractor=self.file_extractor
        ).load_data()

    def _load_dataset(self, dataset_file):
        """Load the RAG dataset from a JSON file"""
        with open(dataset_file, "r") as f:
            # Convert dictionary to object with attribute access
            return json.load(f, object_hook=lambda d: SimpleNamespace(**d))

    def _get_index_persist_path(self, chunk_size, chunk_overlap):
        """Get the directory path for persisting the index"""
        # 使用人類可讀的格式
        index_dir = f"chunk_size{chunk_size}_overlap{chunk_overlap}"
        return os.path.join(self.index_persist_dir, index_dir)

    def _create_or_load_index(self, chunk_size, chunk_overlap):
        """Create a new index or load an existing one based on chunk parameters"""
        index_persist_path = self._get_index_persist_path(chunk_size, chunk_overlap)

        if os.path.exists(index_persist_path):
            print(
                f"Loading existing index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            # Load existing index
            storage_context = StorageContext.from_defaults(
                persist_dir=index_persist_path
            )
            index = load_index_from_storage(storage_context)
        else:
            print(
                f"Creating new index for chunk_size={chunk_size}, chunk_overlap={chunk_overlap}"
            )
            documents = self._load_documents(self.data_directory)
            # Create text splitter with the specified chunk size and overlap
            text_splitter = SentenceSplitter(
                chunk_size=chunk_size, chunk_overlap=chunk_overlap
            )

            # Create vector index
            index = VectorStoreIndex.from_documents(
                documents, transformations=[text_splitter]
            )

            # Persist the index
            index.storage_context.persist(persist_dir=index_persist_path)

        total_num_chunks = len(index.docstore.docs)
        print(f"Loaded index with {total_num_chunks} documents")

        return index

    def _create_query_engine(self, chunk_size, chunk_overlap, top_k):
        """Create a query engine with the specified parameters"""
        # Create or load index
        index = self._create_or_load_index(chunk_size, chunk_overlap)

        # Configure retrievers with the specified top_k
        vector_retriever = index.as_retriever(similarity_top_k=top_k, verbose=True)
        bm25_retriever = BM25Retriever.from_defaults(
            docstore=index.docstore, similarity_top_k=top_k
        )

        # Create fusion retriever
        retriever = QueryFusionRetriever(
            [vector_retriever, bm25_retriever],
            similarity_top_k=top_k,
            num_queries=1,  # set to 1 to disable query generation
            mode="reciprocal_rerank",
            use_async=True,
            verbose=True,
        )

        # Create response synthesizer
        response_synthesizer = get_response_synthesizer()

        # Create and return query engine
        return RetrieverQueryEngine(
            retriever=retriever,
            response_synthesizer=response_synthesizer,
        )

    def _add_eval_row(
        self,
        response,
        dataset_example,
        relevancy_result,
        correctness_result,
        response_time,
    ):
        """Add evaluation results to the DataFrame"""
        if not response.source_nodes:
            print("No response!")
            return

        eval_row = pd.DataFrame(
            [
                {
                    "Query": dataset_example.query,
                    "Response": str(response),
                    "Reference Answer": dataset_example.reference_answer,
                    "Source": response.source_nodes[0].node.text[:1000] + "...",
                    "Relevancy Eval Result": f"{'Pass' if relevancy_result.passing else 'Fail'} \nscore: {relevancy_result.score}",
                    "Relevancy Reasoning": relevancy_result.feedback,
                    "Correctness Eval Result": f"{'Pass' if correctness_result.passing else 'Fail'} \n\nscore: {correctness_result.score}",
                    "Correctness Reasoning": correctness_result.feedback,
                    "Response Time": f"{response_time:.4f}",
                }
            ]
        )

        self.eval_results_df = pd.concat(
            [self.eval_results_df, eval_row], ignore_index=True
        )

    async def _evaluate_engine(
        self, query_engine, dataset_examples: LabelledRagDataExample
    ):
        """Evaluate the query engine on the given examples"""
        # # Limit to first 3 examples for testing
        # dataset_examples = dataset_examples[:3]

        # Initialize tracking variables
        relevancy_total_correct = 0
        correctness_total_correct = 0
        correctness_total_score = 0
        total_response_time = 0

        # Process each example
        for dataset_example in dataset_examples:
            start_time = time.time()
            response = query_engine.query(dataset_example.query)
            response_time = time.time() - start_time

            # Evaluate relevancy and correctness
            relevancy_result = self.relevancy_evaluator.evaluate_response(
                query=dataset_example.query, response=response
            )
            correctness_result = self.correctness_evaluator.evaluate_response(
                query=dataset_example.query,
                response=response,
                reference=dataset_example.reference_answer,
            )

            # Add results to DataFrame
            self._add_eval_row(
                response,
                dataset_example,
                relevancy_result,
                correctness_result,
                response_time,
            )

            # Update totals
            total_response_time += response_time
            if relevancy_result.passing:
                relevancy_total_correct += 1
            if correctness_result.passing:
                correctness_total_correct += 1
                correctness_total_score += correctness_result.score

        return (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            len(dataset_examples),
            total_response_time,
        )

    def evaluate_with_params(self, chunk_size, chunk_overlap, top_k):
        """Run evaluation with the specified parameters"""
        print(
            f"Parameters: chunk_size={chunk_size}, chunk_overlap={chunk_overlap}, top_k={top_k}"
        )

        # Reset results DataFrame
        self.eval_results_df = pd.DataFrame()

        # Create query engine with the specified parameters
        query_engine = self._create_query_engine(chunk_size, chunk_overlap, top_k)

        # Run evaluation
        (
            relevancy_total_correct,
            correctness_total_correct,
            correctness_total_score,
            total_questions,
            total_response_time,
        ) = asyncio.run(self._evaluate_engine(query_engine, self.rag_dataset.examples))

        # Display results
        styled_df = self.eval_results_df.style.set_properties(
            **{"white-space": "pre-wrap"},
        )

        display(styled_df)

        # Calculate scores
        relevancy_score = relevancy_total_correct / total_questions
        correctness_score = correctness_total_score / total_questions
        avg_response_time = total_response_time / total_questions

        # Display summary
        print(
            f"Total Relevancy correct: {relevancy_total_correct} out of {total_questions}, "
            f"score: {relevancy_score}"
        )
        print(
            f"Total Correctness correct: {correctness_total_correct} out of {total_questions}, "
            f"score: {correctness_score}"
        )
        print(f"Average response time: {avg_response_time:.4f} seconds")
        print("===============================================")

        # Add result to summary DataFrame
        self._add_summary_row(
            chunk_size,
            chunk_overlap,
            top_k,
            relevancy_score,
            correctness_score,
            avg_response_time,
        )

        return {
            "relevancy_score": relevancy_score,
            "correctness_score": correctness_score,
            "avg_response_time": avg_response_time,
        }

    def _add_summary_row(
        self,
        chunk_size,
        chunk_overlap,
        top_k,
        relevancy_score,
        correctness_score,
        avg_response_time,
    ):
        """Add a summary row to the summary results DataFrame"""
        summary_row = pd.DataFrame(
            [
                {
                    "Chunk Size": chunk_size,
                    "Chunk Overlap": chunk_overlap,
                    "Top K": top_k,
                    "Relevancy Score": f"{relevancy_score:.4f}",
                    "Correctness Score": f"{correctness_score:.4f}",
                    "Avg Response Time": f"{avg_response_time:.4f}",
                }
            ]
        )

        self.summary_results_df = pd.concat(
            [self.summary_results_df, summary_row], ignore_index=True
        )
        self.summary_results_df.to_csv(
            "summary_results_second_half_last.csv", index=False
        )

    def run_evaluations(self, chunk_sizes, overlaps, top_ks):
        """Run evaluations for multiple parameter combinations"""
        # Reset summary results DataFrame
        # self.summary_results_df = pd.DataFrame()

        # Generate all parameter combinations
        param_combinations = list(itertools.product(chunk_sizes, overlaps, top_ks))
        total_combinations = len(param_combinations)

        print(f"Running evaluations for {total_combinations} parameter combinations...")

        results = {}
        for i, (chunk_size, overlap, top_k) in enumerate(param_combinations):
            print(f"\nEvaluation {i+1}/{total_combinations}")
            # Convert overlap from percentage to absolute value
            chunk_overlap = int(chunk_size * overlap)
            results[(chunk_size, overlap, top_k)] = self.evaluate_with_params(
                chunk_size, chunk_overlap, top_k
            )

        # Display summary table
        self._display_summary_table()

        return results

    def _display_summary_table(self):
        """Display a summary table of all parameter combinations"""
        print("\n--- Summary of All Parameter Combinations ---")

        # Sort the summary results by scores
        sorted_df = self.summary_results_df.sort_values(
            by=["Relevancy Score", "Correctness Score"], ascending=False
        )

        display(sorted_df)

        # Find the best parameter combination
        best_row = sorted_df.iloc[0]
        print(f"\nBest Parameter Combination:")
        print(f"Chunk Size: {best_row['Chunk Size']}")
        print(f"Chunk Overlap: {best_row['Chunk Overlap']}")
        print(f"Top K: {best_row['Top K']}")
        print(f"Relevancy Score: {best_row['Relevancy Score']}")
        print(f"Correctness Score: {best_row['Correctness Score']}")
        print(f"Avg Response Time: {best_row['Avg Response Time']} seconds")

In [3]:
evaluator = RAGEvaluator()

# Define parameter ranges to test
# (512,51,8)~
# (512,25,3)~(512,102,12)
chunk_sizes = [512]
overlaps = [0.2]
top_ks = list(range(13, 16))
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

chunk_sizes = [768, 1024, 1536, 2048]
overlaps = [0.05, 0.1, 0.15, 0.2]
top_ks = list(range(3, 16))
results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

# Define parameter ranges to test
chunk_sizes = [128, 192, 256, 384]
overlaps = [0.05]
top_ks = list(range(3, 16))

results = evaluator.run_evaluations(chunk_sizes, overlaps, top_ks)

Running evaluations for 3 parameter combinations...

Evaluation 1/3
Parameters: chunk_size=512, chunk_overlap=102, top_k=13
Loading existing index for chunk_size=512, chunk_overlap=102
Loaded index with 38 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,**Rewrite** 這是一份關於未來大學學生餐飲服務的資訊手冊，內容涵蓋了不同餐廳的特色菜單、營養師設計的健康餐點，以及推動可持續發展和社會責任的活動。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3....,Pass score: 1.0,"YES. The query asks about the definition of """"學餐資訊"""" (canteen information), and the response provided earlier mentions a ""資訊手冊"" (information handbook) that covers various aspects of university student dining services, including healthy menu options, sustainable development activities, and social responsibility initiatives. This matches the context provided in the new information, which describes different themed restaurants and cafes on campus, each highlighting specific Sustainable Development Goals (SDGs). The new context also mentions various activities and initiatives related to sustainability, social justice, and education, all of which are relevant to the concept of """"學餐資訊"""" (canteen information).",Fail score: 2.0,"The generated answer is relevant to the user query, as it discusses student dining services in a university setting. However, it contains significant mistakes and does not accurately address the question of what ""學餐資訊"" (school meal information) refers to. The reference answer provides a clear definition of school meal information, which is not present in the generated answer.",8.2420
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,**Rewrite** 校內的學生餐廳採用了SDGs目標1：零飢餓、目標9：產業創新和基礎設施、目標11：可持續城市和社區，以及目標12：負責任的消費和生產的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 設立回收獎勵計劃，鼓勵學生積極參與廢物回收。 - 提倡減少浪費與選擇環保產品： - 在校園商店和餐廳推廣使用環保產品，如可重複使用的水瓶和餐具。 - 開展「無塑校園」運動，減少一次性塑膠的使用。 - 舉辦減少浪費的主題活動，如「不浪費日」挑戰，鼓勵學生節約資源。 - 設立可持續發展教育課程： - 開設選修課程，講授可持續發展和負責任消費的概念。 - 將可持續發展納入必修課程，使每個學生都能接觸到相關知識。 # 目標 13：氣候行動 Climate Action - 設立氣候變遷研究中心 - 成立一個跨學科的研究中心，專注於氣候變遷的研究和創新解決方案。 - 鼓勵學生參與研究項目，並提供實習機會。 - 定期舉辦研討會和學術會議，促進校內外的知識交流。...,Pass score: 1.0,"YES. The query mentions that the student cafeteria adopts concepts from SDGs targets 1: Zero Hunger, 9: Industry Innovation and Infrastructure, 11: Sustainable Cities and Communities, and 12: Responsible Consumption and Production. The response confirms this by mentioning that the cafeteria adopts these targets' concepts. In the provided context, it is mentioned that the student cafeteria adopts concepts from SDG target 1 (Zero Hunger) through various sections such as ""每日學餐"" (Daily Student Meal), ""有機農場餐廳"" (Organic Farm Cafe), and ""健康小食堂"" (Healthy Small Food). Additionally, other sections like ""公平貿易餐廳"" (Fair Trade Restaurant), ""和平餐廳"" (Peace Restaurant), and ""社會企業餐廳"" (Social Enterprise Restaurant) also align with the concepts of SDG targets 12 (Responsible Consumption and Production) and 16 (Peace, Justice and Strong Institutions). The cafeteria's adoption of these targets' concepts is further emphasized through the mention of specific initiatives such as reducing food waste, promoting sustainable agriculture, supporting fair trade practices, and fostering social entrepreneurship. Therefore, considering the existing answer was already YES, and the information provided in the new context supports this affirmation, the correct response remains YES.",Pass score: 4.0,"The generated answer is highly relevant to the user query, and it correctly identifies several SDGs targets that are adopted by the student cafeteria. Although it does not mention all the targets mentioned in the reference answer (SDG 3: Good Health and Well-being), it still provides a comprehensive list of adopted targets. The only minor issue is that it omits one target, but this does not significantly impact the overall correctness and relevance of the generated answer.",8.86

Total Relevancy correct: 27 out of 30, score: 0.9
Total Correctness correct: 23 out of 30, score: 3.066666666666667
Average response time: 5.6431 seconds

Evaluation 2/3
Parameters: chunk_size=512, chunk_overlap=102, top_k=14
Loading existing index for chunk_size=512, chunk_overlap=102
Loaded index with 38 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,Eco Eats學餐（強調SDGs目標12：負責任的消費和生產） - (1) 綠色餐廳 - 提供本地採購的食材，減少食物碳足跡。 - 招牌菜：本地風味烤雞、鮮果沙拉。 - (2) 廢物再利用餐廳 - 使用食材的每一部分，推動零廢棄理念。 - 招牌菜：蔬菜莖葉炒飯、柑橘皮蜜餞。 - (3) 環保快餐店 - 所有餐具均為可再生或可降解材料，提倡環保用餐。 - 招牌菜：竹纖維碗拉麵、紙漿盒三明治。 # 3 GOOD HEALTH AND WELL-BEING # 3....,Pass score: 1.0,"YES. The query is ""什麼是學餐資訊？"" (What is school meal information?) and the response provides detailed information about a school's sustainable initiatives, including a green garden cafeteria that serves fresh and healthy meals using locally grown produce, as well as a clean water cafe that promotes personal hygiene education. The context also mentions other SDG-related activities such as community engagement, waste reduction, and job training programs. The new context provided further reinforces the idea that the school is committed to sustainability and social responsibility, with initiatives such as reducing food waste, promoting recycling, and providing free meals for students in need. Therefore, the answer remains YES.",Fail score: 1.0,"The generated answer is completely irrelevant to the user query, as it appears to be a description of a sustainable communities project with no mention of school cafeteria information. The content and structure are unrelated to the reference answer, which provides a clear definition of ""學餐資訊"".",6.6030
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,**Rewrite** 校內的學生餐廳採用了SDGs目標2：零飢餓、目標12：負責任的消費和生產，以及目標3：良好健康與福祉的概念來建立。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 設立回收獎勵計劃，鼓勵學生積極參與廢物回收。 - 提倡減少浪費與選擇環保產品： - 在校園商店和餐廳推廣使用環保產品，如可重複使用的水瓶和餐具。 - 開展「無塑校園」運動，減少一次性塑膠的使用。 - 舉辦減少浪費的主題活動，如「不浪費日」挑戰，鼓勵學生節約資源。 - 設立可持續發展教育課程： - 開設選修課程，講授可持續發展和負責任消費的概念。 - 將可持續發展納入必修課程，使每個學生都能接觸到相關知識。 # 目標 13：氣候行動 Climate Action - 設立氣候變遷研究中心 - 成立一個跨學科的研究中心，專注於氣候變遷的研究和創新解決方案。 - 鼓勵學生參與研究項目，並提供實習機會。 - 定期舉辦研討會和學術會議，促進校內外的知識交流。...,Pass score: 1.0,"Based on the provided context and the original question, I would still answer **YES**. The response to the original question states that the school cafeteria adopted SDGs targets 2 (Zero Hunger), 12 (Responsible Consumption and Production), and 3 (Good Health and Well-being) in its establishment. This is consistent with the new context, which mentions the adoption of these same targets in various initiatives, including the student cafeteria. Therefore, since the information is present in the new context and aligns with the original question, I would still answer **YES**.",Pass score: 4.0,"The generated answer is fully relevant to the user query and contains accurate information about the SDGs targets adopted by the student cafeteria, including target 2 (Zero Hunger), target 12 (Responsible Consumption and Production), and target 3 (Good Health and Well-being). The only difference from the reference answer is that it uses a more concise format, but this does not affect its correctness.",5.8669
2,什麼是 Green Garden 學餐？,# Green Garden 學餐 社區農園餐廳，利用校園社區農園的有機蔬菜，提供新鮮健康的餐點。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6....",Pass score: 1.0,"YES. The response ""什麼是 Green Garden 學餐？"" (What is Green Garden School Lunch?) mentions that Green Garden 學餐 uses school community garden's organic vegetables to provide fresh and healthy meals, which aligns with the context 

Total Relevancy correct: 28 out of 30, score: 0.9333333333333333
Total Correctness correct: 25 out of 30, score: 3.3333333333333335
Average response time: 5.6891 seconds

Evaluation 3/3
Parameters: chunk_size=512, chunk_overlap=102, top_k=15
Loading existing index for chunk_size=512, chunk_overlap=102
Loaded index with 38 documents


...

,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,**Rewrite** 學餐資訊是學校推動的一個計劃，旨在為學生提供健康和均衡的食物選擇。該計劃包括四個不同的餐廳，每個餐廳都強調了一個特定的可持續發展目標（SDGs）。這些餐廳提供了多樣化的菜單選擇，讓學生可以根據自己的需求和偏好進行選擇。 **Rewrite** 學餐資訊計劃的目的是通過提供健康和均衡的食物來促進學生的健康和福祉。該計劃還鼓勵學生了解可持續發展的重要性，並在日常生活中實踐這些價值。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,"YES. The query and response are still in line with the context information provided. The original question ""什麼是學餐資訊?"" (What is school meal information?) was answered affirmatively, and the new context further elaborates on the concept of school meal information as part of a sustainable development goals (SDGs) initiative, which aligns with the previous response.",Pass score: 4.0,"The generated answer is highly relevant to the user query, as it provides detailed information about the school's meal plan and its goals. Although it contains some extra details not present in the reference answer, such as the specific SDGs emphasized by each restaurant, the overall content is accurate and well-organized. The only reason for not giving a perfect score is that the generated answer does not exactly match the reference answer in terms of wording and structure.",9.2754
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,**Rewrite** 校內的學生餐廳採用了零飢餓（Zero Hunger）目標2的概念，提供免費的健康餐點給有需要的學生，並且營養師設計選擇確保健康且均衡。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 將可持續發展納入必修課程，使每個學生都能接觸到相關知識。 # 目標 13：氣候行動 Climate Action - 設立氣候變遷研究中心 - 成立一個跨學科的研究中心，專注於氣候變遷的研究和創新解決方案。 - 鼓勵學生參與研究項目，並提供實習機會。 - 定期舉辦研討會和學術會議，促進校內外的知識交流。 - 推動校園和社區的環保活動 - 舉辦校園清潔日、植樹活動等，提升學生的環保意識。 - 與當地社區合作，組織環保宣傳活動，如步行和騎行倡議。 # 目標 14：水下生物 Life Below Water - 海洋生態系統的研究和保護工作： - 建立與國際或本地海洋保護組織的合作夥伴關係，共同開展海洋生態研究。 - 組織學生參與海洋保護活動，如海岸清潔和海洋生物監測。 - 在學校內部推廣海洋保護的重要性，並設立相關獎學金支持相關研究。...,Pass score: 1.0,"YES The response mentions that the school's student cafeteria adopts the concept of SDG 2 (Zero Hunger) by providing free and healthy meals to students in need. This aligns with the provided context, which also mentions implementing SDG 1: No Poverty and SDG 2: Zero Hunger among other goals.",Pass score: 4.0,"The generated answer is highly relevant to the user query, and it accurately describes how the student cafeteria adopts the concept of SDG Target 2 (Zero Hunger) to provide free healthy meals for students in need. The answer also mentions that a nutritionist designs the menu to ensure healthy and balanced options, which aligns with the reference answer's mention of SDG Target 3 (Good Health and Well-being). Although the generated answer does not explicitly mention SDG Targets 12 and 3 like the reference answer, it still provides a comprehensive understanding of how the student cafeteria adopts sustainable development goals.",7.5770
2,什麼是 Green Garden 學餐？,**Repeat** 社區農園餐廳的菜單中有一道名為「社區農園沙拉》的招牌菜。,社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 - 推動個人衛生教育，所有餐點均在高度衛生環境下製作。 - 招牌菜：無菌沙拉、潔淨便當。 # PEACE, JUSTICE AND STRONG INSTITUTIONS # 6. Peaceful Plates（強調SDGs目標16：和平、正義和強大機構） # (1) 公平貿易餐廳 - 所有食材均來自公平貿易供應商，支持全球正義。 - 招牌菜：公平貿易咖啡、道德巧克力蛋糕。 # (2) 和平餐廳 - 提供來自各地區的和平美食，推動文化交流與理解。 - 招牌菜：和平三明治、全球風味拼盤。 # (3) 社會企業餐廳 - 與社會企業合作，支持社會弱勢群體。 - 招牌菜：慈善義大利麵、公益蔬菜湯。...",Pass score: 1.0,"Based on the provided context and the query ""什麼是 Green Garden 學餐？"", I would answer: YES. The response should provide more specific information about the Green Garden program for promoting healthy eating and reducing food waste, which is mentioned i

Total Relevancy correct: 29 out of 30, score: 0.9666666666666667
Total Correctness correct: 27 out of 30, score: 3.6
Average response time: 8.5678 seconds

Evaluation 23/208
Parameters: chunk_size=768, chunk_overlap=76, top_k=12
Loading existing index for chunk_size=768, chunk_overlap=76
Loaded index with 24 documents


,Query,Response,Reference Answer,Source,Relevancy Eval Result,Relevancy Reasoning,Correctness Eval Result,Correctness Reasoning,Response Time
0,什麼是學餐資訊？,**Rewrite** 這是一份關於學校與全球夥伴建立夥伴關係，推動可持續發展目標實現的資訊。它涵蓋了跨領域研究和合作的機會，並提供相關的學術資源和研究支持，促進社會正義和平等。,學餐資訊是指學校內的學生餐廳提供的餐點資訊，包括不同菜式、營養特點等。,健康學餐（強調SDGs目標3：良好健康與福祉） - (1) 運動員餐廳 - 提供高蛋白、低脂肪的餐點，適合運動後補充能量。 - 招牌菜：健身沙拉、雞胸肉餐盒。 - (2) 心靈健康餐廳 - 提供低糖、低鹽的餐點，支持心理健康。 - 招牌菜：穀物水果碗、香菇豆腐湯。 - (3) 健康果汁吧 - 提供新鮮水果和蔬菜汁，增強免疫力。 - 招牌菜：綠色活力汁、莓果奶昔。...,Pass score: 1.0,"YES. The query ""什麼是學餐資訊？"" (What is school meal information?) and response ""**Rewrite**"" are related to the context provided, which describes a school's initiatives for sustainable development goals (SDGs) and provides details on various programs, including free meals, vocational training, and environmental education. The rewritten response mentions ""學餐資訊"" (school meal information) as part of promoting SDG concepts, specifically focusing on targets 1 (Zero Hunger), 12 (Responsible Consumption and Production), and 3 (Good Health and Well-being).",Fail score: 2.0,"The generated answer is not relevant to the user query, as it talks about a partnership between schools and global partners for sustainable development, which has nothing to do with student cafeteria information. The content of the generated answer also does not match the context of the reference answer or the user query.",7.0914
1,校內的學生餐廳採用哪些SDGs目標的概念來建立？,# 學生餐廳的SDGs概念 學生餐廳採用了未來大學-校園行事曆中提到的SDGs目標3 GOOD HEALTH AND WELL-BEING、11 RESPONSIBLE CONSUMPTION AND PRODUCTION和12 RESPONSIBLE CONSUMPTION AND PRODUCTION的概念。這些目標與健康、可持續城市和社區、以及負責任的消費和生產等方面有關。 在學餐資訊中，學生餐廳採用了SDGs目標3 ZERO HUNGER、12 RESPONSIBLE CONSUMPTION AND PRODUCTION和22 ZERO HUNGER的概念。這些目標與飢餓零化、負責任的消費和生產以及飢餓零化等方面有關。 學生餐廳通過提供健康的小食堂、惜食餐廳和有機農場餐廳等服務，實現了SDGs目標3 GOOD HEALTH AND WELL-BEING。同時，也推動了SDGs目標12 RESPONSIBLE CONSUMPTION AND PRODUCTION和22 ZERO HUNGER的實踐。,根據提供的文本，校內的學生餐廳採用了以下幾個SDGs目標的概念來建立： 1. SDGs目標1：零飢餓 (Zero Hunger) 2. SDGs目標12：負責任的消費和生產 3. SDGs目標3：良好健康與福祉,- 將可持續發展納入必修課程，使每個學生都能接觸到相關知識。 # 目標 13：氣候行動 Climate Action - 設立氣候變遷研究中心 - 成立一個跨學科的研究中心，專注於氣候變遷的研究和創新解決方案。 - 鼓勵學生參與研究項目，並提供實習機會。 - 定期舉辦研討會和學術會議，促進校內外的知識交流。 - 推動校園和社區的環保活動 - 舉辦校園清潔日、植樹活動等，提升學生的環保意識。 - 與當地社區合作，組織環保宣傳活動，如步行和騎行倡議。 # 目標 14：水下生物 Life Below Water - 海洋生態系統的研究和保護工作： - 建立與國際或本地海洋保護組織的合作夥伴關係，共同開展海洋生態研究。 - 組織學生參與海洋保護活動，如海岸清潔和海洋生物監測。 - 在學校內部推廣海洋保護的重要性，並設立相關獎學金支持相關研究。...,Pass score: 1.0,"The existing answer is still YES. The new context provides additional information about the student cafeteria's initiatives and programs that align with specific SDGs targets (2, 12, and 3), which are already mentioned in the original response. The student cafeteria's adoption of concepts from these SDG targets is consistent across both contexts. Therefore, the answer remains YES.",Pass score: 4.0,"The generated answer is relevant to the user query, as it correctly identifies the SDGs targets adopted by the student cafeteria. However, there are some minor mistakes in the answer, such as listing ""SDGs目標3 GOOD HEALTH AND WELL-BEING、11 RESPONSIBLE CONSUMPTION AND PRODUCTION和12 RESPONSIBLE CONSUMPTION AND PRODUCTION"" twice, and also mentioning ""SDGs目標22 ZERO HUNGER"", which is not present in the reference answer. Despite these errors, the overall content of the generated answer is correct and provides a clear explanation of how the student cafeteria adopts SDGs targets.",10.3773
2,什麼是 Green Garden 學餐？,"**Rewrite** Green Garden 學餐是一個創新的學餐方案，旨在提供健康、環保的食物給學生。與社區農園合作，該方案將使用新鮮的農產品，並且減少廢棄和垃圾問題。這樣不僅能夠改善學生的營養，也能夠促進環境保護和可持續發展的理念。 (Note: I rewrote the original answer to fit the new context, which seems to be related to sustainable development and environmental protection.)",社區農園餐廳。,"# SUSTAINABLE CITIES AND COMMUNITIES # 4. Green Garden學餐（強調SDGs目標11：可持續城市和社區） # (1) 社區農園餐廳 - 利用校園社區農園的蔬菜，提供新鮮健康的餐點。 - 招牌菜：社區農園沙拉、蔬菜焗飯。 # (2) 綠建築餐廳 - 餐廳建築採用可再生能源，並設有綠色屋頂。 - 招牌菜：太陽能烤肉、風能披薩。 # CLEAN WATER AND SANITATION # 5. Clean Water Café （強調SDGs目標6：清潔飲水和衛生設施） # (1) 純淨水吧 - 提供純淨飲用水和各類健康飲品，所有水源經過嚴格檢測。 - 招牌菜：純淨水冷泡茶、健康果蔬水。 # (2) 衛生餐廳 -

Total Relevancy correct: 29 out of 30, score: 0.9666666666666667
Total Correctness correct: 20 out of 30, score: 2.6666666666666665
Average response time: 9.3585 seconds

Evaluation 24/208
Parameters: chunk_size=768, chunk_overlap=76, top_k=13
Loading existing index for chunk_size=768, chunk_overlap=76
Loaded index with 24 documents


KeyboardInterrupt: 